# DINOv3 PCA Feature Visualization (DTU Rectified_raw)

这个 notebook 用于在你已有的数据加载 cell 之后，单独执行 DINOv3 特征可视化。

In [ ]:
# DINOv3 特征可视化（单独一个 cell）
# 前置条件：前面的 cell 已经加载了 1600x1200 的 DTU Rectified_raw 图像（RGB 或 BGR）。

from pathlib import Path
import sys
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from sklearn.decomposition import PCA

# 1) 路径设置：优先使用服务器路径，其次回退到当前工程
DINOV3_ROOT = Path('/scr/user/qinglong/projects/upr-mvs01/models/dinov3')
if not DINOV3_ROOT.exists():
    DINOV3_ROOT = Path.cwd() / 'models' / 'dinov3'
if not DINOV3_ROOT.exists():
    raise FileNotFoundError(f'DINOv3 code not found: {DINOV3_ROOT}')

REPO_ROOT = DINOV3_ROOT.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from models.dinov3.extractor import load_dinov3_vit_base, dino_normalization_tensors

# 2) 获取前面 cell 已加载的图像变量（按常见命名自动查找）
candidate_names = ['ref_img', 'ref_image', 'image', 'img', 'rgb', 'image_rgb', 'ref_image_rgb']
image_np = None
for name in candidate_names:
    if name in globals() and isinstance(globals()[name], np.ndarray):
        image_np = globals()[name]
        print(f'Use image variable: {name}, shape={image_np.shape}, dtype={image_np.dtype}')
        break

if image_np is None:
    raise ValueError(
        'No image numpy array found in previous cells. '
        f'Tried {candidate_names}. Please assign one variable, e.g. ref_img.'
    )

# 3) 统一到 RGB uint8
if image_np.ndim != 3 or image_np.shape[2] != 3:
    raise ValueError(f'Expect HxWx3 image, got {image_np.shape}')

if image_np.dtype != np.uint8:
    img = np.clip(image_np, 0, 255).astype(np.uint8)
else:
    img = image_np.copy()

# 如果你的变量是 BGR，这里改为 True
INPUT_IS_BGR = False
if INPUT_IS_BGR:
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

orig_h, orig_w = img.shape[:2]
print(f'Original image size: {orig_w}x{orig_h}')

# 4) 高分辨率输入 + patch 对齐（论文可视化思路）
patch_size = 16
max_side = 1536   # 可改 1024/1536/2048，显存不够就调小
scale = max_side / max(orig_h, orig_w)
target_h = max(patch_size, int(round(orig_h * scale / patch_size)) * patch_size)
target_w = max(patch_size, int(round(orig_w * scale / patch_size)) * patch_size)
print(f'DINO input size (patch-aligned): {target_w}x{target_h}')

img_resized = cv2.resize(img, (target_w, target_h), interpolation=cv2.INTER_AREA)
x = torch.from_numpy(img_resized).permute(2, 0, 1).unsqueeze(0).float() / 255.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DINOV3_WEIGHTS = Path('/scr/user/qinglong/dataset/DINOv3/pre_trained/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth')
if not DINOV3_WEIGHTS.exists():
    # 回退：可通过环境变量 DINOV3_WEIGHTS_FILE 指定
    dinov3_weights_env = os.environ.get('DINOV3_WEIGHTS_FILE', '')
    if dinov3_weights_env:
        DINOV3_WEIGHTS = Path(dinov3_weights_env)
if not DINOV3_WEIGHTS.exists():
    raise FileNotFoundError(f'DINOv3 weights not found: {DINOV3_WEIGHTS}')

model = load_dinov3_vit_base(device=device, weights_file=DINOV3_WEIGHTS, patch_size=patch_size)
mean, std = dino_normalization_tensors(device)
x = (x.to(device) - mean) / std

# 5) 提取 patch token（去 CLS），恢复 H_patch x W_patch x C
with torch.inference_mode():
    tokens = model.get_intermediate_layers(x, n=(11,), reshape=False, norm=True)[0]  # [1, 1+N, C]

patch_tokens = tokens[:, 1:, :]
num_patch_h = target_h // patch_size
num_patch_w = target_w // patch_size
feat = patch_tokens.reshape(1, num_patch_h, num_patch_w, -1)[0].float().cpu().numpy()  # [H_p, W_p, C]
print(f'Patch feature map: {feat.shape}')

# 6) PCA -> 3 维（核心），再归一化
feat_flat = feat.reshape(-1, feat.shape[-1])
pca = PCA(n_components=3, random_state=0)
feat_pca = pca.fit_transform(feat_flat).reshape(num_patch_h, num_patch_w, 3)

feat_pca = feat_pca - feat_pca.min(axis=(0, 1), keepdims=True)
feat_pca = feat_pca / np.clip(feat_pca.max(axis=(0, 1), keepdims=True), 1e-6, None)

# 7) 上采样回原图尺寸 + 平滑（减少 block artifacts）
feat_vis = cv2.resize(feat_pca, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
feat_vis = cv2.GaussianBlur(feat_vis, (5, 5), 0)
feat_vis = np.clip(feat_vis, 0.0, 1.0)

# 8) 展示
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title(f'Input RGB ({orig_w}x{orig_h})')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(feat_vis)
plt.title('DINOv3 Patch Feature PCA (Upsampled + Smoothed)')
plt.axis('off')
plt.tight_layout()
plt.show()

# 可选保存
# out_path = Path('outputs/dinov3_pca_vis.png')
# out_path.parent.mkdir(parents=True, exist_ok=True)
# cv2.imwrite(str(out_path), cv2.cvtColor((feat_vis * 255).astype(np.uint8), cv2.COLOR_RGB2BGR))
# print('Saved:', out_path)
